# CDF datasets: Part 1 pandas cleaning
This notebook implements sections 1.1-1.8 of the supplied cleaning instructions for CDF projects and source posts only. The manually reviewed extraction snapshot is the input, not a substitute for cleaning. Every cleaning operation below uses pandas; original snapshots and narrative evidence are preserved.

Run from the repository root. No network access is needed. The final cells export the existing submission filenames. This notebook covers only the CDF contribution, not the group's complete Part 2 notebook.


In [1]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts/clean_cdf.py").exists() and (ROOT.parent / "scripts/clean_cdf.py").exists():
    ROOT = ROOT.parent  # notebook was launched from notebooks/, repo root is one level up
assert (ROOT / "scripts/clean_cdf.py").exists(), "Run this notebook from the repository root or notebooks/"
PROJECT_FILE = "db-unza26-csc4792-zimba_town_council_cdf_projects.csv"
SOURCE_FILE = "cdf_source_posts.csv"
snapshot = ROOT / "raw/cdf_projects/before_cleaning"
cdf = pd.read_csv(snapshot / PROJECT_FILE, sep="|")
posts = pd.read_csv(snapshot / SOURCE_FILE, sep="|")
for name, df in [("cdf", cdf), ("posts", posts)]:
    print(name, df.shape)
    print(df.head().to_string(index=False))
    df.info()
    print(df.describe(include="all").to_string())


cdf (44, 10)
project_id                              project_name    sector constituency funding_source  funding_amount_zmw    status date_reported                                                                                                                                             description                             source_url
   ZTC-001    2023 CDF cooperative and company loans     other    Mapatizya            CDF                 NaN completed    2023-11-02             Loans were disbursed to 26 cooperatives and companies. The amount is left blank because the source prints the malformed figure K2, 93, 400. https://www.zimbacouncil.gov.zm/?p=911
   ZTC-002      2023 CDF women and youth club grants     other    Mapatizya            CDF           2143990.0 completed    2023-11-02                        K2,143,990 in grants was awarded to 65 women and youth clubs. Completed refers to the reported award, not subsequent activities. https://www.zimbacouncil.gov.zm/?p=911
   ZTC-00

## 1.2 Duplicates: Detect
Count exact duplicates and repeated source URLs, and display the projects sharing URLs. A repeated URL is a candidate for investigation, not proof of duplicated project data.


In [2]:
print("Exact project duplicates:", cdf.duplicated().sum())
print("Repeated project URLs:", cdf.duplicated(subset=["source_url"]).sum())
print("Exact source-post duplicates:", posts.duplicated().sum())
print("Repeated source-post URLs:", posts.duplicated(subset=["url"]).sum())
print(cdf.loc[cdf.duplicated(subset=["source_url"], keep=False),
              ["project_id", "project_name", "source_url"]].to_string(index=False))


Exact project duplicates: 0
Repeated project URLs: 25
Exact source-post duplicates: 0
Repeated source-post URLs: 0
project_id                                                         project_name                              source_url
   ZTC-001                               2023 CDF cooperative and company loans  https://www.zimbacouncil.gov.zm/?p=911
   ZTC-002                                 2023 CDF women and youth club grants  https://www.zimbacouncil.gov.zm/?p=911
   ZTC-003                             Mapatizya 2620 school desks distribution  https://www.zimbacouncil.gov.zm/?p=911
   ZTC-004                            Cikuyu Primary School 1x3 classroom block  https://www.zimbacouncil.gov.zm/?p=982
   ZTC-005                                                   Mbwiko Ward clinic  https://www.zimbacouncil.gov.zm/?p=982
   ZTC-006                                                 Mulamfwu Ward clinic  https://www.zimbacouncil.gov.zm/?p=982
   ZTC-007                            Muziya 

## 1.2 Duplicates: Judge and Act
The 25 repeated project URLs refer to different projects bundled in 19 articles. URL-only deduplication would delete valid records, so the PDF example's one-project-per-article assumption does not apply. Retain distinct projects and separate-date updates; use all original fields except the sequential ID to remove only true project duplicates. Source articles are deduplicated by URL only after confirming no URL has conflicting contents.


In [3]:
project_keys = [c for c in cdf.columns if c != "project_id"]
cdf_clean = cdf.drop_duplicates(subset=project_keys, keep="first").copy()
source_variants = posts.drop_duplicates()
assert not source_variants.duplicated(subset=["url"]).any(), "Review conflicting source versions first"
posts_clean = posts.drop_duplicates(subset=["url"], keep="first").copy()
print("Project duplicates removed:", len(cdf) - len(cdf_clean))
print("Source duplicates removed:", len(posts) - len(posts_clean))


Project duplicates removed: 0
Source duplicates removed: 0


## 1.3 Missing values: Detect
Use `.isnull().sum()` before deciding whether gaps make a record unusable.


In [4]:
print("Project missing values:\n", cdf_clean.isnull().sum())
print("Source missing values:\n", posts_clean.isnull().sum())


Project missing values:
 project_id             0
project_name           0
sector                 0
constituency           2
funding_source         0
funding_amount_zmw    33
status                 4
date_reported          0
description            0
source_url             0
dtype: int64
Source missing values:
 title        0
date         0
body_text    0
url          0
dtype: int64


## 1.3 Missing values: Judge and Act
Funding and status may be targets in later analyses, but useful records remain usable for other questions. Follow the PDF's explicit action table: leave 33 unknown amounts blank, label four unstated statuses `unspecified`, and retain two missing locations without guessing. The illustrative `dropna(status)` snippet is not appropriate here. Description and body text are supporting features and remain unchanged; no rows are dropped for missing targets in this snapshot.


In [5]:
cdf_clean["status"] = cdf_clean["status"].astype("string").str.strip().str.lower().replace("", pd.NA).fillna("unspecified")
# Keep amounts as numeric missing values during analysis, exporting them as blank cells.
funding = pd.to_numeric(cdf_clean["funding_amount_zmw"], errors="coerce")
assert not (cdf_clean["funding_amount_zmw"].notna() & funding.isna()).any(), "Review invalid amounts"
cdf_clean["funding_amount_zmw"] = funding
print(cdf_clean.isnull().sum())
print(cdf_clean["status"].value_counts(dropna=False))


project_id             0
project_name           0
sector                 0
constituency           2
funding_source         0
funding_amount_zmw    33
status                 0
date_reported          0
description            0
source_url             0
dtype: int64
status
planned            20
completed          17
unspecified         4
ongoing             2
near_completion     1
Name: count, dtype: Int64


## 1.4 Outliers: Detect
Compute quartiles and 1.5-IQR bounds on observed funding amounts. Dates and numbers embedded in article text are not measurement columns. Repeated announcements mean these amounts must not be summed as total expenditure.


In [6]:
Q1 = cdf_clean["funding_amount_zmw"].quantile(0.25)
Q3 = cdf_clean["funding_amount_zmw"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = cdf_clean[(cdf_clean["funding_amount_zmw"] < lower) | (cdf_clean["funding_amount_zmw"] > upper)]
print({"Q1": Q1, "Q3": Q3, "IQR": IQR, "lower": lower, "upper": upper})
print(outliers[["project_name", "funding_amount_zmw", "source_url"]].to_string(index=False))


{'Q1': np.float64(700000.0), 'Q3': np.float64(1922924.31), 'IQR': np.float64(1222924.31), 'lower': np.float64(-1134386.465), 'upper': np.float64(3757310.7750000004)}
Empty DataFrame
Columns: [project_name, funding_amount_zmw, source_url]
Index: []


## 1.4 Outliers: Judge and Act
No values are flagged in this snapshot, so no source investigations or corrections are necessary. Keep all observed values. If a later dataset produces flags, stop to review their saved source HTML/URLs; do not delete an extreme value simply because it is large.


In [7]:
assert outliers.empty, "Review flagged amounts against their sources before continuing"
assert (cdf_clean["funding_amount_zmw"].dropna() > 0).all()
print("No outliers removed or corrected.")


No outliers removed or corrected.


## 1.5 Optional text cleaning: Judge and Act
Preserve original narrative evidence and add separate analysis features. Remove HTML first, then case-fold, remove punctuation through word tokenization and remove an explicit small stopword list. Regex tokenization uses the allowed toolkit without NLTK downloads; negations such as `not` are retained. This is a documented regex equivalent of the optional example, not a claim to use NLTK's tokenizer or full English stopword corpus.


In [8]:
stop_words = set("a an the and or of to in on at for from by with as is are was were be been being it its this that these those".split())
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = text.casefold()
    tokens = re.findall(r"\b\w+\b", text)
    return " ".join(token for token in tokens if token not in stop_words)

cdf_clean["description_clean"] = cdf_clean["description"].apply(clean_text)
posts_clean["body_text_clean"] = posts_clean["body_text"].apply(clean_text)
print(cdf_clean[["description", "description_clean"]].head().to_string(index=False))


                                                                                                                                            description                                                                                                           description_clean
            Loans were disbursed to 26 cooperatives and companies. The amount is left blank because the source prints the malformed figure K2, 93, 400.                loans disbursed 26 cooperatives companies amount left blank because source prints malformed figure k2 93 400
                       K2,143,990 in grants was awarded to 65 women and youth clubs. Completed refers to the reported award, not subsequent activities.                    k2 143 990 grants awarded 65 women youth clubs completed refers reported award not subsequent activities
Distribution of 2,620 desks procured under 2022 and 2023 CDF was launched. The printed amount K 2, 887,051 is normalized by removing spaces and commas. distribution 2 620 d

## 1.6 Standardize types and categories
Convert dates with `pd.to_datetime`, funding with `pd.to_numeric`, and strip/lowercase categories. Stop on invalid nonblank values rather than silently lose evidence. CSV files do not store pandas dtypes, so dates serialize in ISO format and numeric gaps serialize as blanks.


In [9]:
for frame, column in [(cdf_clean, "date_reported"), (posts_clean, "date")]:
    parsed = pd.to_datetime(frame[column], errors="coerce")
    assert not (frame[column].notna() & parsed.isna()).any(), "Review invalid dates"
    frame[column] = parsed
cdf_clean["funding_amount_zmw"] = pd.to_numeric(cdf_clean["funding_amount_zmw"], errors="coerce")
for column in ["status", "sector"]:
    cdf_clean[column] = cdf_clean[column].astype("string").str.strip().str.lower()
assert cdf_clean["status"].isin(["planned", "ongoing", "near_completion", "completed", "unspecified"]).all()
assert cdf_clean["sector"].isin(["education", "health", "water_sanitation", "agriculture", "infrastructure", "other"]).all()
assert cdf_clean["source_url"].notna().all() and posts_clean["url"].notna().all()
for name, df in [("cdf_clean", cdf_clean), ("posts_clean", posts_clean)]:
    print(name, df.shape)
    df.info()
    print(df.describe(include="all").to_string())


cdf_clean (44, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   project_id          44 non-null     object        
 1   project_name        44 non-null     object        
 2   sector              44 non-null     string        
 3   constituency        42 non-null     object        
 4   funding_source      44 non-null     object        
 5   funding_amount_zmw  11 non-null     float64       
 6   status              44 non-null     string        
 7   date_reported       44 non-null     datetime64[ns]
 8   description         44 non-null     object        
 9   source_url          44 non-null     object        
 10  description_clean   44 non-null     object        
dtypes: datetime64[ns](1), float64(1), object(7), string(2)
memory usage: 3.9+ KB
       project_id                            project_name     sector

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   title            29 non-null     object        
 1   date             29 non-null     datetime64[ns]
 2   body_text        29 non-null     object        
 3   url              29 non-null     object        
 4   body_text_clean  29 non-null     object        
dtypes: datetime64[ns](1), object(4)
memory usage: 1.3+ KB
                                                                                                 title                           date                                                                                                                                                                                                                                                                                                                                               

## 1.7 Export and verify
Export only the two CDF submission files, with their original filenames, pipe separators and no index. Reload both exports to validate their dimensions and columns. Compare this explicitly implemented notebook pipeline against the standalone pandas cleaner to prevent divergent outputs.


In [10]:
import importlib.util
spec = importlib.util.spec_from_file_location("cdf_cleaning_script", ROOT / "scripts/clean_cdf.py")
cleaner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cleaner)
expected_cdf, _ = cleaner.clean(cdf, True)
expected_posts, _ = cleaner.clean(posts, False)
pd.testing.assert_frame_equal(cdf_clean, expected_cdf)
pd.testing.assert_frame_equal(posts_clean, expected_posts)
for filename, df in [(PROJECT_FILE, cdf_clean), (SOURCE_FILE, posts_clean)]:
    target = ROOT / "data" / filename
    exported = df.to_csv(sep="|", index=False, date_format="%Y-%m-%d", na_rep="").encode("utf-8-sig")
    # Do not rewrite identical exports, which may be open in a spreadsheet app.
    if not target.exists() or target.read_bytes() != exported:
        target.write_bytes(exported)
    reopened = pd.read_csv(target, sep="|")
    assert reopened.shape == df.shape
    assert list(reopened.columns) == list(df.columns)
    assert "|" in target.read_text(encoding="utf-8-sig").splitlines()[0]
    print(filename, reopened.shape)
print("Notebook and standalone pandas pipeline agree.")


db-unza26-csc4792-zimba_town_council_cdf_projects.csv (44, 11)


cdf_source_posts.csv (29, 5)
Notebook and standalone pandas pipeline agree.


## 1.8 Handoff
**Projects:** 44 announcement rows and 11 columns; no true duplicates removed. Unknown amounts remain blank in 33 rows and locations in two; four unstated statuses are `unspecified`. No IQR outliers were flagged or removed. Original descriptions accompany the new `description_clean` feature.

**Source posts:** 29 article rows and five columns; no duplicates removed and no missing original fields. Dates are standardized and original narrative text accompanies `body_text_clean`. Numeric IQR analysis does not apply to this text evidence table. Export round trips and agreement with the standalone pandas pipeline are checked above.

Supporting evidence: `raw/cdf_projects/before_cleaning/`, `raw/cdf_projects/post_*.html`, and `docs/cdf_cleaning/README.md`. This preserves legitimate multi-project articles and is not a unique-project expenditure register.
